# Part 1 — Notebook 01: MadGraph Process Setup and LHE Generation

## Pedagogical Goal & Overview

Welcome to **Part 1 — Notebook 01** of the experimental High-Energy Physics (HEP) Monte Carlo training series.

In this notebook, you will execute the first hands-on step of event generation:
$$\text{Process Definition} \rightarrow \text{MadGraph Process/Run Cards} \rightarrow \text{Parton-Level Events (LHE)} \rightarrow \text{Event Record Inspection}$$

### Primary Pedagogical Process
We study proton-proton production of a $Z$ boson recoiling against a hard matrix-element jet, with the $Z$ boson forced to decay to a bottom-quark pair:
```text
p p > z j, z > b b~
```

### Full Monte Carlo Event Pipeline Context
It is critical to understand where this notebook fits in the experimental physics chain:
$$\underbrace{\text{Hard Scattering (ME)} \rightarrow \text{Resonance Decay}}_{\text{Notebook 01 (MadGraph LHE)}} \rightarrow \text{Parton Shower} \rightarrow \text{Hadronization} \rightarrow \text{Detector Sim} \rightarrow \text{Reconstruction} \rightarrow \text{Analysis}$$

In this notebook, we operate purely at the **parton-level matrix element (ME)** stage.

## Step 1: Environment Setup & Compiler Verification

Before executing event generators, we verify our computing environment in Google Colab. MadGraph requires Python 3 and a C++/Fortran compiler (`gfortran`) to compile matrix-element code.

In [ ]:
import os
import sys
import subprocess

print("=== Environment Verification ===")
print(f"Python version: {sys.version.split()[0]}")

# Verify gfortran compiler
try:
    gfortran_info = subprocess.check_output(["gfortran", "--version"]).decode('utf-8').split('\n')[0]
    print(f"GFortran compiler: {gfortran_info}")
except Exception as e:
    print(f"WARNING: gfortran check failed ({e}). MadGraph requires gfortran to compile matrix elements.")

content_dir = "/content" if os.path.exists("/content") else os.getcwd()
mg5_dir = os.path.join(content_dir, "MG5_aMC")
print(f"Target MadGraph directory: {mg5_dir}")


> [!IMPORTANT]
> ### Self-Reflection & Method Checkpoint 1.1
> 1. **Coding Question**: Why do we use `subprocess.check_output` instead of `os.system` when checking system software versions in Python?
> 2. **Environment Question**: What happens if `gfortran` is missing when MadGraph attempts to generate matrix elements (`generate p p > z j`)?


## Step 2: Download and Install MadGraph5_aMC@NLO

We download the official MadGraph5_aMC@NLO release archive (`MG5_aMC_v3.5.16.tar.gz`), extract it to `/content/MG5_aMC`, and confirm that the executable `./bin/mg5_aMC` exists.

In [ ]:
mg5_tar = "MG5_aMC_v3.5.16.tar.gz"
mg5_url = "https://launchpad.net/mg5amcnlo/3.0/3.7.x/+download/MG5_aMC_v3.5.16.tar.gz"

if not os.path.exists(mg5_dir):
    print("Downloading MadGraph5_aMC@NLO v3.5.16...")
    !wget -q {mg5_url} -O {mg5_tar}
    print("Extracting archive...")
    !tar -xzf {mg5_tar}
    if os.path.exists("MG5_aMC_v3.5.16") and not os.path.exists(mg5_dir):
        os.rename("MG5_aMC_v3.5.16", mg5_dir)
else:
    print(f"MadGraph5 already installed at {mg5_dir}")

mg5_exe = os.path.join(mg5_dir, "bin", "mg5_aMC")
assert os.path.exists(mg5_exe), f"ERROR: MadGraph executable not found at {mg5_exe}"
print(f"SUCCESS: Verified MadGraph executable at {mg5_exe}")


## Step 3: Integrated Physics Foundations — Standard Model, Boosted Z, & Event Pipeline

### 1. Minimal Standard Model Orientation
- **Quarks**: $u, d, c, s, t, b$. Bottom quarks ($b, \bar{b}$) carry color charge and weak isospin.
- **Gauge Bosons**: Photon ($\gamma$), $W^\pm$, $Z^0$, gluon ($g$). The $Z$ boson is a neutral heavy electroweak gauge boson ($m_Z \approx 91.2\text{ GeV}$).

### 2. $Z$ Production and Hadronic Decay
At the LHC ($\sqrt{s} = 13\text{ TeV}$), $Z$ bosons are produced via quark-antiquark annihilation ($q\bar{q} \to Z$). $Z \to b\bar{b}$ has a large branching ratio (~15.1%), but is dominated by background from pure Quantum Chromodynamics (QCD) multijet events ($pp \to jj$).

### 3. Boosted Topology
By requiring a hard recoiling jet ($j$) at matrix element level with $p_T^j > 150\text{ GeV}$, the $Z$ boson recoils with high transverse momentum ($p_T^Z > 150\text{ GeV}$). Relativistic boost collimates the decay partons ($b, \bar{b}$):
$$\Delta R_{b\bar{b}} \approx \frac{2m_Z}{p_T^Z}$$

### 4. Generator Truth vs. Reconstructed Data
- **Generator Truth**: The exact, simulated four-momenta of matrix-element partons ($PID = \pm 5, status = 1$). Unavailable in real recorded collider data!
- **Reconstructed Jet**: An algorithmic proxy created from calorimeter energy deposits or tracks, NOT identical to an initial parton.
- **LHE Record**: Stores matrix-element truth objects ($b, \bar{b}, Z$), NOT detector-reconstructed jets.

---

> [!IMPORTANT]
> ### Exercise 3: Physics Pipeline & Truth Identification
> Answer the following questions based on the pipeline above:
> 1. Which stages of the event pipeline ($\text{ME} \rightarrow \text{Decay} \rightarrow \text{Shower} \rightarrow \text{Hadronization} \rightarrow \text{Detector} \rightarrow \text{Reconstruction} \rightarrow \text{Analysis}$) are performed by MadGraph in this notebook?
> 2. Why is a status 1 $b$ quark in an LHE file considered "generator truth" rather than a reconstructed $b$-jet?


In [ ]:
# Exercise 3 Reference Solution:
# 1. MadGraph performs Hard Scattering (ME calculation) and Resonance Decay (Z -> b b~).
#    Parton Showering, Hadronization, Detector Simulation, and Jet Reconstruction belong to later stages (Pythia/Delphes/FastJet).
# 2. Status 1 b-quarks in LHE are bare matrix-element partons directly produced from Z decay.
#    They have not undergone gluon radiation, hadronization into B-hadrons, or calorimeter energy clustering.

pipeline_answer = "Hard Scattering (ME) and Resonance Decay (Z -> b b~)"
truth_answer = "LHE b-quarks are bare matrix-element truth partons before showering and detector simulation"

print("=== Exercise 3 Reference Answers Verified ===")
print(f"Pipeline stages present: {pipeline_answer}")
print(f"Truth explanation: {truth_answer}")


## Step 4: Inspect & Review MadGraph Cards

MadGraph uses two cards to define generation:
1. **Process Card (`cards/zbbj_proc_card.dat`)**:
   ```text
   import model sm
   generate p p > z j, z > b b~
   output Zbbj_LO
   launch Zbbj_LO
   ```
   The comma `, z > b b~` specifies forced decay of the intermediate $Z$ boson to $b\bar{b}$.

2. **Run Card (`cards/zbbj_run_card.dat`)**:
   Specifies center-of-mass energy ($\sqrt{s} = 13\text{ TeV}$ with `ebeam1 = 6500`, `ebeam2 = 6500`), random seed (`iseed = 42`), event count (`nevents = 1000`), and kinematic cuts (`ptj = 150`, `ptZmin = 150`).

---

> [!IMPORTANT]
> ### Exercise 4: Card Parameter Review & Understanding
> Inspect the card parameters above and answer the following questions:
> 1. What collision center-of-mass energy ($\sqrt{s}$) is configured by `ebeam1 = 6500` and `ebeam2 = 6500`?
> 2. What does the comma in `generate p p > z j, z > b b~` do?
> 3. Why do we set `ptj = 150` and `ptZmin = 150` in the run card? What physics topology does this cut enforce?


In [ ]:
cards_dir = "cards"
os.makedirs(cards_dir, exist_ok=True)

proc_card_path = os.path.join(cards_dir, "zbbj_proc_card.dat")
proc_text = "import model sm\ngenerate p p > z j, z > b b~\noutput Zbbj_LO\nlaunch Zbbj_LO\n"
with open(proc_card_path, "w") as f:
    f.write(proc_text)

# Exercise 4 Reference Answers:
com_energy_gev = 13000   # 6500 + 6500 = 13000 GeV = 13 TeV
decay_syntax_meaning = "The comma specifies forced decay of intermediate Z -> b b~"
boost_cut_purpose = "ptj=150 and ptZmin=150 force high-pT Z recoil against matrix-element jet"

print(f"Verified process card at {proc_card_path}")
print("Card Parameter Review Solution:")
print(f"1. CoM Energy: {com_energy_gev} GeV ({com_energy_gev/1000:.0f} TeV)")
print(f"2. Decay Syntax: {decay_syntax_meaning}")
print(f"3. Boost Cut Rationale: {boost_cut_purpose}")


## Step 5a: Test Run — 10 Events & Execution Log Reading

Before executing a full production run of 1000 events, we first run a fast **10-event test run** (`set nevents 10`).

### Reading the Execution Log
As MadGraph runs in non-interactive batch mode, read the live log output to observe:
1. Feynman diagram generation and C++ matrix element compilation.
2. Phase space integration and total cross-section calculation ($\sigma$ in picobarn, pb).
3. Unweighted event generation and compression into `.lhe.gz` format.

In [ ]:
mg5_batch_test = "run_zbbj_test.mg5"
batch_test_content = "import model sm\ngenerate p p > z j, z > b b~\noutput Zbbj_test\nlaunch Zbbj_test\nset nevents 10\nset iseed 42\nset ebeam1 6500\nset ebeam2 6500\nset ptj 150\nset ptZmin 150\ndone\n"

with open(mg5_batch_test, "w") as f:
    f.write(batch_test_content)

print("Executing 10-event test run and streaming stdout log...")
!{mg5_exe} {mg5_batch_test}

test_lhe = "Zbbj_test/Events/run_01/unweighted_events.lhe.gz"
assert os.path.exists(test_lhe), f"ERROR: Test LHE file missing at {test_lhe}"
print(f"\nSUCCESS: Test run completed! Verified 10-event LHE at {test_lhe}")


> [!IMPORTANT]
> ### Self-Reflection & Log Checkpoint 5.1
> 1. Read the stdout execution log above. What cross section ($\sigma$) was calculated by MadGraph for $pp \to Z+j, Z \to b\bar{b}$ with $p_T^j > 150\text{ GeV}$?
> 2. How many Feynman diagrams were generated for this hard scattering process?


## Step 5b: Production Run — 1000 Events for Analysis

Now that the 10-event test run has successfully completed and we verified the execution log, we run the **main production run of 1000 events** (`set nevents 1000`).

The output file `Zbbj_LO/Events/run_01/unweighted_events.lhe.gz` will be parsed and analyzed in Notebook 02.

In [ ]:
mg5_batch_prod = "run_zbbj_prod.mg5"
batch_prod_content = "import model sm\ngenerate p p > z j, z > b b~\noutput Zbbj_LO\nlaunch Zbbj_LO\nset nevents 1000\nset iseed 42\nset ebeam1 6500\nset ebeam2 6500\nset ptj 150\nset ptZmin 150\ndone\n"

with open(mg5_batch_prod, "w") as f:
    f.write(batch_prod_content)

print("Executing 1000-event production run (~1-2 mins)...")
!{mg5_exe} {mg5_batch_prod}

lhe_path = "Zbbj_LO/Events/run_01/unweighted_events.lhe.gz"
assert os.path.exists(lhe_path), f"ERROR: Missing production LHE file at {lhe_path}"
file_size_kb = os.path.getsize(lhe_path) / 1024

print(f"\n==========================================")
print(f"SUCCESS: 1000-event production run completed!")
print(f"Production LHE: {lhe_path}")
print(f"File Size: {file_size_kb:.2f} KB")
print(f"==========================================")


## Step 6: Les Houches Event (LHE) Structure Inspection

An LHE file stores event records in XML format.

### Key Record Columns:
- `PID`: $+5 = b$, $-5 = \bar{b}$, $23 = Z$, $21 = g$, $1..4 = u,d,c,s$.
- `Status`: `-1` (incoming parton), `+1` (outgoing final-state parton), `+2` (intermediate decayed resonance).
- `Mother1, Mother2`: Indices pointing to parent particles.
- `Px, Py, Pz, E, Mass`: Particle four-momentum components in GeV.

In [ ]:
import gzip

lhe_path = "Zbbj_LO/Events/run_01/unweighted_events.lhe.gz"

print(f"Inspecting production LHE file: {lhe_path}")
with gzip.open(lhe_path, "rt") as f:
    lines = f.readlines()

print(f"Total lines in LHE file: {len(lines)}")

event_lines = []
recording = False
for line in lines:
    if "<event>" in line:
        recording = True
    if recording:
        event_lines.append(line)
    if "</event>" in line:
        break

print("\n--- Sample First LHE Event Block ---")
print("".join(event_lines))


> [!IMPORTANT]
> ### Exercise 6: LHE Particle Record Identification
> In the output of the first `<event>` block above, answer the following questions:
> 1. What are the PIDs of the incoming initial-state partons ($status = -1$)?
> 2. Find the intermediate $Z$ boson ($PID = 23$). What is its status code?
> 3. Find the final-state bottom quarks ($PID = 5$ and $PID = -5$). What are their status codes?
> 4. Where is the Monte Carlo event weight $w_i$ located in the `<event>` header line?
